<a href="https://colab.research.google.com/github/madhavsantthosh/taekwondo-kinematics-tracker/blob/main/Taekwondo_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install mediapipe -q


!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.0 MB/s eta 0:00:00


actual code

VIDEO HERE

In [ ]:
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO
)
detector = vision.PoseLandmarker.create_from_options(options)

input_video_path = "kick_input.mp4"
cap = cv2.VideoCapture(input_video_path)

if not cap.isOpened():
    print(f"Error: Could not open {input_video_path}. Did you upload it?")
else:
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    output_video_path = "kick_output.avi"
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    print("Processing video, rendering HUD, and logging data... Please wait.")

    telemetry_data = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        frame_timestamp_ms = int((frame_count / fps) * 1000)

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        detection_result = detector.detect_for_video(mp_image, frame_timestamp_ms)

        if detection_result.pose_landmarks:
            landmarks = detection_result.pose_landmarks[0]


            connections = [
                (11, 12), (11, 23), (12, 24), (23, 24),
                (11, 13), (13, 15), (12, 14), (14, 16),
                (23, 25), (25, 27), (24, 26), (26, 28)
            ]
            for p1, p2 in connections:
                x1, y1 = int(landmarks[p1].x * frame_width), int(landmarks[p1].y * frame_height)
                x2, y2 = int(landmarks[p2].x * frame_width), int(landmarks[p2].y * frame_height)
                cv2.line(frame, (x1, y1), (x2, y2), (255, 127, 0), 2) # Blue lines

            for landmark in landmarks:
                x = int(landmark.x * frame_width)
                y = int(landmark.y * frame_height)
                cv2.circle(frame, (x, y), 5, (0, 0, 255), -1)
                cv2.circle(frame, (x, y), 6, (255, 255, 255), 1)

            # --- MATH & LOGIC ---
            hip = [landmarks[24].x, landmarks[24].y]
            knee = [landmarks[26].x, landmarks[26].y]
            ankle = [landmarks[28].x, landmarks[28].y]
            knee_angle = calculate_angle(hip, knee, ankle)

            guard_drop = landmarks[16].y > landmarks[12].y

            if knee_angle < 75:
                feedback_text = "STATUS: OPTIMAL CHAMBER"
                status_color = (0, 255, 0)
            elif guard_drop:
                feedback_text = "WARNING: GUARD DROPPED"
                status_color = (0, 0, 255)
            else:
                feedback_text = "STATUS: EXTENDING"
                status_color = (255, 255, 255)

            # --- HUD OVERLAY ---
            overlay = frame.copy()
            cv2.rectangle(overlay, (10, 10), (320, 95), (0, 0, 0), -1)
            cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

            cv2.putText(frame, "BIOMECHANICAL HUD v1.0", (20, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1, cv2.LINE_AA)
            cv2.putText(frame, f"KNEE ANGLE: {int(knee_angle)} deg", (20, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.putText(frame, feedback_text, (20, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, status_color, 2, cv2.LINE_AA)


            telemetry_data.append({
                "Frame": frame_count,
                "Timestamp_ms": frame_timestamp_ms,
                "Knee_Angle_Deg": round(knee_angle, 2)
            })

        out.write(frame)

    cap.release()
    out.release()


    df = pd.DataFrame(telemetry_data)
    df.sort_values(by="Frame", inplace=True)
    df.to_csv("kick_data.csv", index=False)

    #CHROMEBOOK COVERSION
    print("Converting video format for ChromeOS compatibility...")
    os.system("ffmpeg -i kick_output.avi -c:v libx264 -pix_fmt yuv420p chromebook_kick.mp4 -y -loglevel quiet")

    print("SUCCESS! Your project is complete. Download 'chromebook_kick.mp4' and 'kick_data.csv'.")

Processing video, rendering HUD, and logging data... Please wait.
Converting video format for ChromeOS compatibility...
SUCCESS! Your project is complete. Download 'chromebook_kick.mp4' and 'kick_data.csv'.
